# Aegis Medical AI: Fine-Tuning Llama 3.2 on Doctor-Patient Conversations

This notebook fine-tunes **Llama 3.2 3B Instruct** using QLoRA on:
1. **Clinical Conversations**: `nisten/opus5-5-doctor-patient-conversations-all-human-diseases` (2,194 clinician-verified diseases).
2. **Helpful & Harmless Alignment**: `Anthropic/hh-rlhf` safety principles.

### Recommended Environment: Free Google Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU)

In [ ]:
# 1. Install required packages
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

In [ ]:
# 2. Check GPU
!nvidia-smi

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
DATASET_ID = "nisten/opus5-5-doctor-patient-conversations-all-human-diseases"
OUTPUT_DIR = "./aegis-medical-lora"

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

In [ ]:
# 3. Load & Format Clinical Dataset
dataset = load_dataset(DATASET_ID, split="train")
print("Total diseases in dataset:", len(dataset))

def format_conversation(example):
    conv = example.get("conversation", [])
    if isinstance(conv, list) and len(conv) > 0:
        return {"messages": conv}
    name = example.get("name", "Condition")
    desc = example.get("description", "")
    return {
        "messages": [
            {"role": "user", "content": f"Explain clinical details for {name}"},
            {"role": "assistant", "content": desc}
        ]
    }

formatted = dataset.map(format_conversation, remove_columns=dataset.column_names)
print("Sample record:", formatted[0])

In [ ]:
# 4. 4-bit Quantization Config & Model Loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# LoRA target modules for Llama 3
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. Train Model
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted.select(range(1000)), # Fast 1,000-sample run
    peft_config=peft_config,
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()

In [ ]:
# 6. Save Fine-Tuned LoRA Adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete! Adapter saved to:", OUTPUT_DIR)

### 7. Exporting to Local Ollama

To use your fine-tuned weights inside Aegis:
1. Download `aegis-medical-lora` folder.
2. In your local Aegis project terminal, run:
```powershell
ollama create aegis-medical -f backend/data/medical/Modelfile
```
3. Set `AI_MODEL=aegis-medical` in `.env`!